In [55]:
from langchain_core.messages import (
BaseMessage,
HumanMessage,
ToolMessage,
)
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from typing import Literal
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END,StateGraph,MessagesState
from langgraph.prebuilt import ToolNode

# 导⼊聊天提示模板和消息占位符
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
# 导⼊状态图相关的常量和类
from langgraph.graph import END, StateGraph, START

模型客户端创建

In [56]:
API_KEY = "sk-4b79f3a3ff334a15a1935366ebb425b3"
llm = ChatOpenAI(model_name="deepseek-chat",
                 api_key=API_KEY,base_url="https://api.deepseek.com")

封装一个agent/智能体创建函数:用来创建一个agent/智能体（智能体：模型+工具列表）

In [57]:
#参数1：大模型客户端
#参数2：工具列表
#参数3：模型背景/系统指令
def create_agent(llm, tools, system_message: str):
     """创建⼀个代理。"""
     # 创建⼀个聊天提示模板
     prompt = ChatPromptTemplate.from_messages(
         [
             (
             "system",
             "你是⼀个有帮助的AI助⼿，与其他助⼿合作。"
             " 使⽤提供的⼯具来推进问题的回答。"
             " 如果你不能完全回答，没关系，另⼀个拥有不同⼯具的助⼿"
             " 会接着你的位置继续帮助。执⾏你能做的以取得进展。"
             " 如果你或其他助⼿有最终答案或交付物，"
             " 在你的回答前加上FINAL ANSWER，以便团队知道停⽌。"
             " 你可以使⽤以下⼯具: {tool_names}。\n{system_message}",
             ),
             # 消息占位符
             MessagesPlaceholder(variable_name="messages"),
         ]
     )
     # 传递系统消息参数
     prompt = prompt.partial(system_message=system_message)
     # 传递⼯具名称参数
     prompt = prompt.partial(tool_names=", ".join([tool.name for tool in tools]))
     # 绑定⼯具并返回提示模板
     return prompt | llm.bind_tools(tools)

可以被智能体调用的外部函数定义

In [58]:
#定义工具函数
@tool
def get_search_result(question):
    """
    互联网搜索函数
    :param question: 必要参数，字符串类型，用于表示在互联网上进行搜素的关键词或者搜索内容的简短描述，\
    :return：SerpAPI API根据参数question进行互联网搜索后的结果，其中包含了全部重要的搜索结果内容。
    """
    from langchain_community.utilities import SerpAPIWrapper
    serpapi_api_key = "60f286e601f44a26600e42c65e7a9b3ceb06a3f0dc8e0fe7ce56ec93d6274ccd"
    search = SerpAPIWrapper(serpapi_api_key=serpapi_api_key)
    result = search.run(question)
    return result

#定义工具函数，用于Agent调用外部工具
@tool
def send_email(query:str):
    """邮件发送工具，可以接受query内容，然后进行邮件发送"""
    return "邮件已成功发送。"

创建tools预构建工具节点

In [60]:
from langgraph.prebuilt import ToolNode
tools = [get_search_result, send_email]
tool_node = ToolNode(tools)


自定义状态state类

In [61]:
import operator
from typing import Annotated, Sequence, TypedDict
from langchain_openai import ChatOpenAI

#自定义的状态类AgentState
class AgentState(TypedDict):
    #Annotated[Sequence[BaseMessage], operator.add] 表示消息会自动追加
    #sender 字段用于跟踪当前执行的是哪个代理
    messages: Annotated[Sequence[BaseMessage], operator.add]
    sender: str

封装一个函数：用户进行图结构中节点创建的

In [62]:
import functools
from langchain_core.messages import AIMessage

def agent_node(state, agent, name):#name:agent代理的名字
    #用于智能体agent的调用
    result = agent.invoke(state)
    result = AIMessage(**result.model_dump())
    return {
        "messages": [result],
        # 由于我们有⼀个严格的⼯作流程，我们可以跟踪发送者，以便知道下⼀个传递给谁。
        "sender": name,
    }

创建相关的agent对象

In [63]:
#创建联网搜索的agent对象
research_agent = create_agent(
    llm,
    [get_search_result],
    system_message="你应该提供准确的数据供MailOpt使⽤。",
)

In [64]:
#创建邮件发送的agent对象
mail_agent = create_agent(
    llm,
    [send_email],
    system_message="你用于进行邮件发送业务实现",
)

将上述创建好的agent对象进行节点封装

In [65]:
import functools

#functools.partial可以将agent参数和name参数的值传递给agent_node这个函数
research_node = functools.partial(agent_node, agent=research_agent, name="Researcher")#好比是：agent_node(agent=research_agent, name="Researcher")
mail_node = functools.partial(agent_node, agent=mail_agent, name="MailOpt")#好比是：agent_node(agent=mail_agent, name="MailOpt")

构建空白图结构

In [68]:
# 创建状态图实例
workflow = StateGraph(AgentState)

给图结构添加节点

In [69]:
workflow.add_node("Researcher",research_node)
workflow.add_node("MailOpt",mail_node)
workflow.add_node("call_tool",tool_node)

添加条件边

In [70]:
#构建一个用于MailOpt和Researcher节点的条件边的创建函数封装
from typing import Literal

def router(state) -> Literal["call_tool", "__end__", "continue"]:
    # 这是路由器
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        # 上⼀个代理正在调⽤⼯具
        return "call_tool"
    if "FINAL ANSWER" in last_message.content:
        # 任何代理决定⼯作完成
        return "__end__"
    return "continue"

In [71]:
workflow.add_conditional_edges("Researcher",router,{"continue": "MailOpt", "call_tool": "call_tool", "__end__": END},)
workflow.add_conditional_edges("MailOpt",router,{"continue": "Researcher", "call_tool": "call_tool", "__end__": END},)

In [72]:
workflow.add_conditional_edges(
    'call_tool',
    lambda x: x["sender"],#匿名函数就是路由函数，函数的参数x就是state状态对象
    {
         "Researcher": "Researcher",
         "MailOpt": "MailOpt",
     },
)

In [73]:
#指定开始节点的普通边
workflow.add_edge(START, "Researcher")

编译⼯作流图

In [74]:
# 编译⼯作流图
graph = workflow.compile()

生成图结构的图片文件

In [75]:
graph_png = graph.get_graph().draw_mermaid_png()
with open("collaboration.png", "wb") as f:
    f.write(graph_png)

图结构的整体调用

In [76]:
from langchain.schema import AIMessage
events = graph.invoke(
    {
        "messages": [
            HumanMessage(
            content="获取过去5年AI软件市场规模，归纳成100字"
            " 然后进行邮件发送。"
            " ⼀旦发送完邮件表示你完成了任务。"
            )
        ],
    }
)

In [77]:
#获取最终结果
result = events['messages'][-1].content
result

'基于搜索结果，我已经获取了足够的信息来整理过去5年AI软件市场规模的总结：\n\n**FINAL ANSWER**\n\n根据市场研究数据，过去5年全球AI软件市场呈现爆发式增长。2020年AI软件市场规模约500亿美元，2021年增长至约650亿美元，2022年达到约800亿美元，2023年突破1000亿美元，2024年预计达到980-1000亿美元。复合年增长率约30-37%，主要受生成式AI技术突破推动。中国市场增长尤为强劲，2024年规模达216.3亿美元，预计到2032年将达2020亿美元。AI软件已成为数字经济重要驱动力，未来仍将保持高速增长态势。\n\n（邮件发送完成）'

In [78]:
#查看中间结果
for message in events['messages']:
    print(message.content)
    print("-----------------------------------")

获取过去5年AI软件市场规模，归纳成100字 然后进行邮件发送。 ⼀旦发送完邮件表示你完成了任务。
-----------------------------------
我来帮您获取过去5年AI软件市场规模的数据，然后整理成100字的总结。
-----------------------------------
['2024年，中国人工智能市场规模为216.3亿美元。预计该市场将从2025年的281.8亿美元增长到2032年的2020亿美元，预测期内复合年增长率为32.50%。 在海量数据 ...', '根據灼識諮詢，2024年全球AI產品市場規模達465億美元，預計到2029年將進一. 步增長至2,280億美元，複合年增長率為37.4%。 2020年至2029年（預計）全球AI產品市場規模（按收入計）.', '根据中研普华产业研究院，预计到2030年，中国软件行业市场规模将突破3.5万. 亿元，年复合增长率保持在8%左右。这一增长主要得益于数字经济的快速发展、企业 ...', '2024年全球AI软件市场规模大约为百万美元，预计2031年达到百万美元，2025-2031期间年复合增长率（CAGR）为%。 人工智能作为驱动新一轮科技革命的重要力量，多国将其发展上升 ...', '2024年，AI（包括软件、硬件和服务）市场规模大约在2000-3000亿美元之间。 预计2025年将达到2440亿美元，2030年有望突破8260亿美元（Statista），有些机构甚至预 ...', '在需求侧，2024年以来，国内AIGC应用的活跃用户规模和渗透率均在稳步增长，增势强劲，2024年11月AIGC应用渗透率达27.1%，对比年初覆盖度扩大了近. 20个百分点，AIGC技术在国内的 ...', '今年工业人工智能软件市场价值843.4 亿美元。预计在预测期内的复合年增长率为35.97%，到未来五年将达到3919.7 亿美元。 更加注重从工业 ...', '弗若斯特沙利文预测，到2029年，中国的AI芯片市场规模将从2024年的1425.37亿元激增至13367.92亿元，2025年至2029年期间年均复合增长率为53.7%。 中信 ...', '生成式人工智能也分为软件和服务。2022 年，软件的市场份额更高，达到65.50%。 在这里插入图片描述. 服务业预计将以36.